In [2]:
import pandas as pd
import xarray as xr
import numpy as np
import os
import glob
import gc
import sys
import subprocess
import shutil

# --- CONFIGURATION ---
# Auto-detect paths for local structure
if os.path.exists("../data"):
    SOURCE_DIR = "../data/raw/nasa_sst_data"
    OUTPUT_DIR = "../data/processed"
else:
    SOURCE_DIR = "data/raw/nasa_sst_data"
    OUTPUT_DIR = "data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)
# Output as a DATASET FOLDER (safest for large data)
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "nasa_sst_data_combined.parquet")

# COARSEN FACTOR:
# 1 = Keep Original 9km Resolution (Best Balance)
COARSEN_FACTOR = 1 

def install_deps():
    try:
        import h5netcdf
        import pyarrow
    except ImportError:
        print("Installing dependencies...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "h5netcdf", "pyarrow"])

def extract_time(ds):
    for attr in ['time_coverage_start', 'period_start_time', 'start_time']:
        if attr in ds.attrs:
            return pd.to_datetime(ds.attrs[attr])
    return None

def process_single_file(filepath):
    try:
        with xr.open_dataset(filepath, engine='h5netcdf') as ds:
            timestamp = extract_time(ds)
            if timestamp is None:
                try:
                    date_str = os.path.basename(filepath).split('.')[1].split('_')[0] 
                    timestamp = pd.to_datetime(date_str)
                except:
                    return None

            var = next((v for v in ds.data_vars if v == 'sst'), None)
            if not var: 
                var = next((v for v in ds.data_vars if 'sst' in v.lower()), None)
            
            if not var: return None

            # Coarsen logic
            if COARSEN_FACTOR > 1:
                ds_processed = ds.coarsen(lat=COARSEN_FACTOR, lon=COARSEN_FACTOR, boundary='trim').mean()
            else:
                ds_processed = ds 
            
            df = ds_processed[var].to_dataframe().reset_index()
            df = df.dropna(subset=[var])
            df['time'] = timestamp
            
            # Optimization: 9km data is still big, use float32
            df['lat'] = df['lat'].astype('float32')
            df['lon'] = df['lon'].astype('float32')
            df[var] = df[var].astype('float32')
            
            df = df.rename(columns={var: 'sst'})
            
            return df
    except Exception as e:
        print(f"   [FAIL] Error processing {os.path.basename(filepath)}: {e}")
        return None

def main():
    print(f"--- Starting SST Conversion (Input: 9km | Factor: {COARSEN_FACTOR}) ---")
    install_deps()
    
    if not os.path.exists(SOURCE_DIR):
        print(f"Error: {SOURCE_DIR} not found.")
        return

    # Cleanup old run
    if os.path.exists(OUTPUT_FILE):
        if os.path.isdir(OUTPUT_FILE): shutil.rmtree(OUTPUT_FILE)
        else: os.remove(OUTPUT_FILE)
    os.makedirs(OUTPUT_FILE, exist_ok=True)

    nc_files = sorted(glob.glob(os.path.join(SOURCE_DIR, "*.nc")))
    print(f"Found {len(nc_files)} files. Writing iteratively...")
    
    total_rows = 0
    for i, f in enumerate(nc_files):
        df_chunk = process_single_file(f)
        
        if df_chunk is not None and not df_chunk.empty:
            # Write chunk to disk immediately
            chunk_name = f"part_{i:04d}.parquet"
            df_chunk.to_parquet(os.path.join(OUTPUT_FILE, chunk_name), index=False)
            total_rows += len(df_chunk)
            del df_chunk
        
        if (i+1) % 10 == 0:
            print(f"   Processed {i+1}/{len(nc_files)}... (Rows: {total_rows:,})")
            gc.collect()

    print(f"\n--- SUCCESS! ---")
    print(f"Data saved to: {OUTPUT_FILE}")
    print(f"Total Rows: {total_rows:,}")

if __name__ == "__main__":
    main()

--- Starting SST Conversion (Input: 9km | Factor: 1) ---
Found 282 files. Writing iteratively...
   Processed 10/282... (Rows: 50,507,378)
   Processed 20/282... (Rows: 101,482,595)
   Processed 30/282... (Rows: 151,925,212)
   Processed 40/282... (Rows: 203,316,228)
   Processed 50/282... (Rows: 254,753,766)
   Processed 60/282... (Rows: 305,733,824)
   Processed 70/282... (Rows: 355,969,052)
   Processed 80/282... (Rows: 407,297,969)
   Processed 90/282... (Rows: 457,839,354)
   Processed 100/282... (Rows: 508,934,282)
   Processed 110/282... (Rows: 560,925,111)
   Processed 120/282... (Rows: 611,566,761)
   Processed 130/282... (Rows: 662,080,876)
   Processed 140/282... (Rows: 712,885,840)
   Processed 150/282... (Rows: 763,449,636)
   Processed 160/282... (Rows: 814,478,801)
   Processed 170/282... (Rows: 866,169,945)
   Processed 180/282... (Rows: 917,295,729)
   Processed 190/282... (Rows: 967,784,989)
   Processed 200/282... (Rows: 1,019,288,746)
   Processed 210/282... (Rows: 

In [6]:
import pandas as pd
import numpy as np
import os
import glob
try:
    import pyarrow.parquet as pq
except ImportError:
    print("Pyarrow not found. Installing...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow"])
    import pyarrow.parquet as pq

# --- CONFIGURATION ---
# Auto-detect paths
if os.path.exists("../data/processed"):
    DATA_DIR = "../data/processed"
elif os.path.exists("data/processed"):
    DATA_DIR = "data/processed"
else:
    DATA_DIR = "."

SST_FILE = os.path.join(DATA_DIR, "nasa_sst_data_combined.parquet")
CHL_FILE = os.path.join(DATA_DIR, "chlorophyll_data_combined.parquet")

def inspect_folder_dataset(folder_path, var_name):
    """
    RAM-Safe inspection for multi-file Parquet datasets (folders).
    """
    print(f"\n" + "="*50)
    print(f"Checking Dataset: {os.path.basename(folder_path)}")
    print("="*50)
    
    # Get all partition files
    files = sorted(glob.glob(os.path.join(folder_path, "*.parquet")))
    
    if not files:
        print("[FAIL] No .parquet files found in directory.")
        return

    print(f"Structure:        Partitioned Dataset (Folder)")
    print(f"Partitions:       {len(files)}")

    # 1. Total Rows Scan (Metadata only - Fast & Low RAM)
    total_rows = 0
    print("Scanning metadata for total count...")
    for f in files:
        meta = pq.read_metadata(f)
        total_rows += meta.num_rows
    print(f"Total Rows:       {total_rows:,}")

    # 2. Value & Time Check (Sampling)
    # We load the first file, a middle file, and the last file to get a representative sample
    sample_indices = [0, len(files)//2, len(files)-1]
    sample_files = [files[i] for i in sample_indices]
    
    print(f"Sampling {len(sample_files)} partitions for validity check...")
    
    dfs = []
    for f in sample_files:
        dfs.append(pd.read_parquet(f))
    
    sample_df = pd.concat(dfs)
    
    # 3. Time Analysis (from sample)
    if 'time' in sample_df.columns:
        # Check start from first file and end from last file specifically
        # (Assuming files are sorted chronologically which they should be from the conversion script)
        start_df = pd.read_parquet(files[0], columns=['time'])
        end_df = pd.read_parquet(files[-1], columns=['time'])
        
        start_time = pd.to_datetime(start_df['time']).min()
        end_time = pd.to_datetime(end_df['time']).max()
        
        print(f"Time Range:       {start_time.date()} to {end_time.date()}")
        
        # Check for gap (simple heuristic using sample)
        months_in_sample = pd.to_datetime(sample_df['time']).dt.to_period('M').nunique()
        print(f"Sampled Months:   {months_in_sample} unique months found in sample.")
    else:
        print("[FAIL] 'time' column missing!")

    # 4. Coordinate Check
    if 'lat' in sample_df.columns:
        lat_min, lat_max = sample_df['lat'].min(), sample_df['lat'].max()
        lon_min, lon_max = sample_df['lon'].min(), sample_df['lon'].max()
        print(f"Latitude Range:   {lat_min:.2f} to {lat_max:.2f}")
        print(f"Longitude Range:  {lon_min:.2f} to {lon_max:.2f}")

    # 5. Value Validity
    if var_name in sample_df.columns:
        stats = sample_df[var_name]
        print(f"\nVariable '{var_name}' Stats (Sampled):")
        print(f"   Min:  {stats.min():.4f}")
        print(f"   Max:  {stats.max():.4f}")
        print(f"   Mean: {stats.mean():.4f}")
        
        if var_name == 'sst':
            if stats.min() < -5 or stats.max() > 45:
                print(">> [WARNING] SST values seem unrealistic (< -5C or > 45C).")
            else:
                print(">> [PASS] SST values are within realistic ocean ranges.")
        elif var_name == 'chl_conc':
            if stats.min() < 0:
                print(">> [FAIL] Negative Chlorophyll values found.")
            else:
                print(">> [PASS] Chlorophyll values look standard.")
    else:
        print(f"[FAIL] Column '{var_name}' not found.")


def inspect_single_file(filepath, var_name):
    """Fallback for single file parquet."""
    print(f"\n" + "="*50)
    print(f"Checking File: {os.path.basename(filepath)}")
    print("="*50)
    if not os.path.exists(filepath):
        print(f"[NOTE] File not found: {filepath}")
        return

    try:
        # Read just metadata/columns first if possible, but for single file pandas reads all.
        # Use pyarrow to peek if installed
        meta = pq.read_metadata(filepath)
        print(f"Total Rows:       {meta.num_rows:,}")
        
        # Read subset if huge
        if meta.num_rows > 1_000_000:
            print("Large single file detected. Reading sample...")
            # Unfortunately pd.read_parquet doesn't support 'nrows', but we can read columns
            df = pd.read_parquet(filepath) # Might crash if REALLY huge
        else:
            df = pd.read_parquet(filepath)
            
        if var_name in df.columns:
             print(f"Variable '{var_name}' found. Range: {df[var_name].min():.2f} to {df[var_name].max():.2f}")
        
    except Exception as e:
        print(f"[ERROR] Could not read file: {e}")


def main():
    print("--- Starting RAM-Safe Data Audit ---")
    
    # Check SST
    if os.path.isdir(SST_FILE):
        inspect_folder_dataset(SST_FILE, 'sst')
    else:
        inspect_single_file(SST_FILE, 'sst')
    
    # Check Chlorophyll
    if os.path.isdir(CHL_FILE):
        inspect_folder_dataset(CHL_FILE, 'chl_conc')
    else:
        # Check if it exists as a file
        inspect_single_file(CHL_FILE, 'chl_conc')

    print("\n--- Audit Complete ---")

if __name__ == "__main__":
    main()

--- Starting RAM-Safe Data Audit ---

Checking Dataset: nasa_sst_data_combined.parquet
Structure:        Partitioned Dataset (Folder)
Partitions:       282
Scanning metadata for total count...
Total Rows:       1,448,077,004
Sampling 3 partitions for validity check...
Time Range:       2002-08-06 to 2026-01-01


C:\Users\tejsr\AppData\Local\Temp\ipykernel_36904\3175019467.py:78: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months_in_sample = pd.to_datetime(sample_df['time']).dt.to_period('M').nunique()


Sampled Months:   3 unique months found in sample.
Latitude Range:   -81.88 to 89.96
Longitude Range:  -179.96 to 179.96

Variable 'sst' Stats (Sampled):
   Min:  -1.8000
   Max:  39.9950
   Mean: 17.2624
>> [PASS] SST values are within realistic ocean ranges.

Checking Dataset: chlorophyll_data_combined.parquet
Structure:        Partitioned Dataset (Folder)
Partitions:       282
Scanning metadata for total count...
Total Rows:       1,261,557,482
Sampling 3 partitions for validity check...
Time Range:       2002-08-06 to 2026-01-01


C:\Users\tejsr\AppData\Local\Temp\ipykernel_36904\3175019467.py:78: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  months_in_sample = pd.to_datetime(sample_df['time']).dt.to_period('M').nunique()


Sampled Months:   3 unique months found in sample.
Latitude Range:   -78.38 to 83.88
Longitude Range:  -179.96 to 179.96

Variable 'chl_conc' Stats (Sampled):
   Min:  0.0012
   Max:  86.1855
   Mean: 0.4159
>> [PASS] Chlorophyll values look standard.

--- Audit Complete ---


In [5]:
import pandas as pd
import xarray as xr
import numpy as np
import os
import glob
import gc
import sys
import subprocess
import shutil

# --- CONFIGURATION ---
# Auto-detect paths
if os.path.exists("../data"):
    SOURCE_DIR = "../data/raw/nasa_chl_data"
    OUTPUT_DIR = "../data/processed"
else:
    SOURCE_DIR = "data/raw/nasa_chl_data"
    OUTPUT_DIR = "data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)
# Output as a DATASET FOLDER to match SST structure and save RAM
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "chlorophyll_data_combined.parquet")

# COARSEN FACTOR:
# 1 = Original Resolution (9km) - Matches your SST data
COARSEN_FACTOR = 1

def install_deps():
    try:
        import h5netcdf
        import pyarrow
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "h5netcdf", "pyarrow"])

def extract_time(ds):
    for attr in ['time_coverage_start', 'period_start_time', 'start_time']:
        if attr in ds.attrs:
            return pd.to_datetime(ds.attrs[attr])
    return None

def process_single_file(filepath):
    try:
        with xr.open_dataset(filepath, engine='h5netcdf') as ds:
            timestamp = extract_time(ds)
            if timestamp is None:
                try:
                    date_str = os.path.basename(filepath).split('.')[1].split('_')[0] 
                    timestamp = pd.to_datetime(date_str)
                except:
                    return None

            # Identify Variable (chlor_a)
            var = next((v for v in ds.data_vars if 'chlor' in v.lower() or 'chl' in v.lower()), None)
            if not var: return None

            # Coarsen logic
            if COARSEN_FACTOR > 1:
                ds_processed = ds.coarsen(lat=COARSEN_FACTOR, lon=COARSEN_FACTOR, boundary='trim').mean()
            else:
                ds_processed = ds 
            
            # Flatten
            df = ds_processed[var].to_dataframe().reset_index()
            df = df.dropna(subset=[var])
            df['time'] = timestamp
            
            # Optimize types (Float32 saves 50% RAM)
            df['lat'] = df['lat'].astype('float32')
            df['lon'] = df['lon'].astype('float32')
            df[var] = df[var].astype('float32')
            
            df = df.rename(columns={var: 'chl_conc'})
            
            return df
    except Exception as e:
        print(f"   [FAIL] Error processing {os.path.basename(filepath)}: {e}")
        return None

def main():
    print(f"--- Starting Chlorophyll Conversion (Resolution Factor: {COARSEN_FACTOR}) ---")
    install_deps()
    
    if not os.path.exists(SOURCE_DIR):
        print(f"Error: {SOURCE_DIR} not found.")
        return

    # Clean up previous output
    if os.path.exists(OUTPUT_FILE):
        if os.path.isdir(OUTPUT_FILE): shutil.rmtree(OUTPUT_FILE)
        else: os.remove(OUTPUT_FILE)
    os.makedirs(OUTPUT_FILE, exist_ok=True)

    nc_files = sorted(glob.glob(os.path.join(SOURCE_DIR, "*.nc")))
    print(f"Found {len(nc_files)} files. Writing iteratively...")
    
    total_rows = 0
    for i, f in enumerate(nc_files):
        df_chunk = process_single_file(f)
        
        if df_chunk is not None and not df_chunk.empty:
            # Write chunk to disk immediately
            chunk_name = f"part_{i:04d}.parquet"
            df_chunk.to_parquet(os.path.join(OUTPUT_FILE, chunk_name), index=False)
            total_rows += len(df_chunk)
            del df_chunk
        
        if (i+1) % 10 == 0:
            print(f"   Processed {i+1}/{len(nc_files)}... (Rows: {total_rows:,})")
            gc.collect()

    print(f"\n--- SUCCESS! ---")
    print(f"Data saved to: {OUTPUT_FILE}")
    print(f"Total Rows: {total_rows:,}")

if __name__ == "__main__":
    main()

--- Starting Chlorophyll Conversion (Resolution Factor: 1) ---
Found 282 files. Writing iteratively...
   Processed 10/282... (Rows: 45,932,697)
   Processed 20/282... (Rows: 91,354,681)
   Processed 30/282... (Rows: 135,579,989)
   Processed 40/282... (Rows: 180,449,503)
   Processed 50/282... (Rows: 225,379,164)
   Processed 60/282... (Rows: 269,991,161)
   Processed 70/282... (Rows: 315,723,919)
   Processed 80/282... (Rows: 360,992,794)
   Processed 90/282... (Rows: 404,835,984)
   Processed 100/282... (Rows: 449,511,966)
   Processed 110/282... (Rows: 494,238,457)
   Processed 120/282... (Rows: 538,602,072)
   Processed 130/282... (Rows: 584,627,987)
   Processed 140/282... (Rows: 629,819,991)
   Processed 150/282... (Rows: 673,834,795)
   Processed 160/282... (Rows: 718,505,011)
   Processed 170/282... (Rows: 763,598,371)
   Processed 180/282... (Rows: 808,608,020)
   Processed 190/282... (Rows: 854,838,065)
   Processed 200/282... (Rows: 900,527,761)
   Processed 210/282... (Row

In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import os
import glob
import gc
import sys
import subprocess
import shutil

# --- CONFIGURATION ---
if os.path.exists("../data"):
    SOURCE_DIR = "../data/raw/nasa_par_data"
    OUTPUT_DIR = "../data/processed"
else:
    SOURCE_DIR = "data/raw/nasa_par_data"
    OUTPUT_DIR = "data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "nasa_par_data_combined.parquet")
COARSEN_FACTOR = 1 

def extract_time(ds):
    for attr in ['time_coverage_start', 'period_start_time', 'start_time']:
        if attr in ds.attrs:
            return pd.to_datetime(ds.attrs[attr])
    return None

def process_single_file(filepath):
    try:
        with xr.open_dataset(filepath, engine='h5netcdf') as ds:
            timestamp = extract_time(ds)
            if timestamp is None:
                date_str = os.path.basename(filepath).split('.')[1].split('_')[0] 
                timestamp = pd.to_datetime(date_str)

            # Look for 'par' specifically
            var = next((v for v in ds.data_vars if v == 'par'), None)
            if not var: return None

            ds_processed = ds if COARSEN_FACTOR <= 1 else ds.coarsen(lat=COARSEN_FACTOR, lon=COARSEN_FACTOR, boundary='trim').mean()
            
            df = ds_processed[var].to_dataframe().reset_index()
            df = df.dropna(subset=[var])
            df['time'] = timestamp
            
            df['lat'] = df['lat'].astype('float32')
            df['lon'] = df['lon'].astype('float32')
            df[var] = df[var].astype('float32')
            
            return df.rename(columns={var: 'par'})
    except Exception as e:
        print(f"   [FAIL] {os.path.basename(filepath)}: {e}")
        return None

def main():
    print(f"--- Starting PAR Conversion ---")
    if not os.path.exists(SOURCE_DIR): return
    if os.path.exists(OUTPUT_FILE):
        shutil.rmtree(OUTPUT_FILE) if os.path.isdir(OUTPUT_FILE) else os.remove(OUTPUT_FILE)
    os.makedirs(OUTPUT_FILE, exist_ok=True)

    nc_files = sorted(glob.glob(os.path.join(SOURCE_DIR, "*.nc")))
    total_rows = 0
    for i, f in enumerate(nc_files):
        df_chunk = process_single_file(f)
        if df_chunk is not None and not df_chunk.empty:
            df_chunk.to_parquet(os.path.join(OUTPUT_FILE, f"part_{i:04d}.parquet"), index=False)
            total_rows += len(df_chunk)
            del df_chunk
        if (i+1) % 10 == 0:
            print(f"   Processed {i+1}/{len(nc_files)}... (Rows: {total_rows:,})")
            gc.collect()
    print(f"--- SUCCESS! Total Rows: {total_rows:,} ---")

if __name__ == "__main__":
    main()

--- Starting PAR Conversion ---
   Processed 10/282... (Rows: 52,883,787)
   Processed 20/282... (Rows: 105,853,617)
   Processed 30/282... (Rows: 158,797,237)
   Processed 40/282... (Rows: 212,217,273)
   Processed 50/282... (Rows: 265,636,606)
   Processed 60/282... (Rows: 318,313,786)
   Processed 70/282... (Rows: 371,498,190)
   Processed 80/282... (Rows: 424,815,592)
   Processed 90/282... (Rows: 477,836,438)
   Processed 100/282... (Rows: 531,352,698)
   Processed 110/282... (Rows: 584,886,014)
   Processed 120/282... (Rows: 637,652,132)
   Processed 130/282... (Rows: 691,062,054)
   Processed 140/282... (Rows: 744,241,926)
   Processed 150/282... (Rows: 797,168,522)
   Processed 160/282... (Rows: 851,123,712)
   Processed 170/282... (Rows: 905,168,505)
   Processed 180/282... (Rows: 958,542,066)
   Processed 190/282... (Rows: 1,012,320,475)
   Processed 200/282... (Rows: 1,066,332,669)
   Processed 210/282... (Rows: 1,120,635,482)
   Processed 220/282... (Rows: 1,175,262,499)
  

In [1]:
import pandas as pd
import xarray as xr
import os
import glob
import gc
import shutil

# --- CONFIGURATION ---
if os.path.exists("../data"):
    SOURCE_DIR = "../data/raw/nasa_kd490_data"
    OUTPUT_DIR = "../data/processed"
else:
    SOURCE_DIR = "data/raw/nasa_kd490_data"
    OUTPUT_DIR = "data/processed"

OUTPUT_FILE = os.path.join(OUTPUT_DIR, "nasa_kd490_data_combined.parquet")

def process_single_file(filepath):
    try:
        with xr.open_dataset(filepath, engine='h5netcdf') as ds:
            # Timestamp logic
            date_str = os.path.basename(filepath).split('.')[1].split('_')[0] 
            timestamp = pd.to_datetime(date_str)

            # Look for Kd_490
            var = next((v for v in ds.data_vars if 'kd_490' in v.lower()), None)
            if not var: return None

            df = ds[var].to_dataframe().reset_index()
            df = df.dropna(subset=[var])
            df['time'] = timestamp
            df['lat'] = df['lat'].astype('float32')
            df['lon'] = df['lon'].astype('float32')
            df[var] = df[var].astype('float32')
            
            return df.rename(columns={var: 'kd_490'})
    except Exception as e:
        print(f"   [FAIL] {os.path.basename(filepath)}: {e}")
        return None

def main():
    print(f"--- Starting Kd_490 Conversion ---")
    if os.path.exists(OUTPUT_FILE): shutil.rmtree(OUTPUT_FILE)
    os.makedirs(OUTPUT_FILE, exist_ok=True)

    nc_files = sorted(glob.glob(os.path.join(SOURCE_DIR, "*.nc")))
    total_rows = 0
    for i, f in enumerate(nc_files):
        df_chunk = process_single_file(f)
        if df_chunk is not None:
            df_chunk.to_parquet(os.path.join(OUTPUT_FILE, f"part_{i:04d}.parquet"), index=False)
            total_rows += len(df_chunk)
        if (i+1) % 10 == 0:
            print(f"   Processed {i+1}/{len(nc_files)}... (Rows: {total_rows:,})")
            gc.collect()
    print(f"--- SUCCESS! Total Rows: {total_rows:,} ---")

if __name__ == "__main__":
    main()

--- Starting Kd_490 Conversion ---
   Processed 10/282... (Rows: 45,932,697)
   Processed 20/282... (Rows: 91,354,681)
   Processed 30/282... (Rows: 135,579,989)
   Processed 40/282... (Rows: 180,449,503)
   Processed 50/282... (Rows: 225,379,164)
   Processed 60/282... (Rows: 269,991,161)
   Processed 70/282... (Rows: 315,723,919)
   Processed 80/282... (Rows: 360,992,794)
   Processed 90/282... (Rows: 404,835,984)
   Processed 100/282... (Rows: 449,511,966)
   Processed 110/282... (Rows: 494,238,457)
   Processed 120/282... (Rows: 538,602,072)
   Processed 130/282... (Rows: 584,627,987)
   Processed 140/282... (Rows: 629,819,991)
   Processed 150/282... (Rows: 673,834,795)
   Processed 160/282... (Rows: 718,505,011)
   Processed 170/282... (Rows: 763,598,371)
   Processed 180/282... (Rows: 808,608,020)
   Processed 190/282... (Rows: 854,838,066)
   Processed 200/282... (Rows: 900,527,762)
   Processed 210/282... (Rows: 944,748,351)
   Processed 220/282... (Rows: 989,128,014)
   Proce

In [2]:
import pandas as pd
import xarray as xr
import numpy as np
import os
import glob
import gc
import sys
import subprocess
import shutil

# --- CONFIGURATION ---
# Auto-detect paths for local structure
if os.path.exists("../data"):
    SOURCE_DIR = "../data/raw/nasa_poc_data"
    OUTPUT_DIR = "../data/processed"
else:
    SOURCE_DIR = "data/raw/nasa_poc_data"
    OUTPUT_DIR = "data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)
# Output as a DATASET FOLDER
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "nasa_poc_data_combined.parquet")

# COARSEN FACTOR:
# 1 = Keep Original 9km Resolution (Recommended for POC to see coastal details)
COARSEN_FACTOR = 1 

def install_deps():
    try:
        import h5netcdf
        import pyarrow
    except ImportError:
        print("Installing dependencies...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "h5netcdf", "pyarrow"])

def extract_time(ds):
    # Try standard attributes
    for attr in ['time_coverage_start', 'period_start_time', 'start_time']:
        if attr in ds.attrs:
            return pd.to_datetime(ds.attrs[attr])
    return None

def process_single_file(filepath):
    try:
        with xr.open_dataset(filepath, engine='h5netcdf') as ds:
            timestamp = extract_time(ds)
            
            # Fallback: Extract date from filename (e.g., AQUA_MODIS.20220101...)
            if timestamp is None:
                try:
                    date_str = os.path.basename(filepath).split('.')[1].split('_')[0] 
                    timestamp = pd.to_datetime(date_str)
                except:
                    return None

            # --- POC SPECIFIC VARIABLE SEARCH ---
            # NASA POC variable is usually 'poc', but sometimes capitalized
            var = next((v for v in ds.data_vars if v.lower() == 'poc'), None)
            
            if not var: 
                # print(f"Variable 'poc' not found in {os.path.basename(filepath)}")
                return None

            # Coarsen logic (Optional)
            if COARSEN_FACTOR > 1:
                ds_processed = ds.coarsen(lat=COARSEN_FACTOR, lon=COARSEN_FACTOR, boundary='trim').mean()
            else:
                ds_processed = ds 
            
            # Convert to DataFrame
            df = ds_processed[var].to_dataframe().reset_index()
            df = df.dropna(subset=[var])
            df['time'] = timestamp
            
            # Optimization: float32
            df['lat'] = df['lat'].astype('float32')
            df['lon'] = df['lon'].astype('float32')
            df[var] = df[var].astype('float32')
            
            # Standardize Column Name to 'poc' for the Zipper Merge later
            df = df.rename(columns={var: 'poc'})
            
            return df
    except Exception as e:
        print(f"   [FAIL] Error processing {os.path.basename(filepath)}: {e}")
        return None

def main():
    print(f"--- Starting POC Conversion (Input: 9km | Factor: {COARSEN_FACTOR}) ---")
    install_deps()
    
    if not os.path.exists(SOURCE_DIR):
        print(f"Error: {SOURCE_DIR} not found.")
        return

    # Cleanup old run
    if os.path.exists(OUTPUT_FILE):
        if os.path.isdir(OUTPUT_FILE): shutil.rmtree(OUTPUT_FILE)
        else: os.remove(OUTPUT_FILE)
    os.makedirs(OUTPUT_FILE, exist_ok=True)

    nc_files = sorted(glob.glob(os.path.join(SOURCE_DIR, "*.nc")))
    print(f"Found {len(nc_files)} files. Writing iteratively...")
    
    total_rows = 0
    for i, f in enumerate(nc_files):
        df_chunk = process_single_file(f)
        
        if df_chunk is not None and not df_chunk.empty:
            # Write chunk to disk immediately
            chunk_name = f"part_{i:04d}.parquet"
            df_chunk.to_parquet(os.path.join(OUTPUT_FILE, chunk_name), index=False)
            total_rows += len(df_chunk)
            del df_chunk
        
        # Memory Management
        if (i+1) % 10 == 0:
            print(f"   Processed {i+1}/{len(nc_files)}... (Rows: {total_rows:,})")
            gc.collect()

    print(f"\n--- SUCCESS! ---")
    print(f"Data saved to: {OUTPUT_FILE}")
    print(f"Total Rows: {total_rows:,}")

if __name__ == "__main__":
    main()

--- Starting POC Conversion (Input: 9km | Factor: 1) ---
Found 282 files. Writing iteratively...
   Processed 10/282... (Rows: 45,917,152)
   Processed 20/282... (Rows: 91,322,780)
   Processed 30/282... (Rows: 135,530,578)
   Processed 40/282... (Rows: 180,385,043)
   Processed 50/282... (Rows: 225,298,366)
   Processed 60/282... (Rows: 269,899,300)
   Processed 70/282... (Rows: 315,619,005)
   Processed 80/282... (Rows: 360,875,423)
   Processed 90/282... (Rows: 404,702,519)
   Processed 100/282... (Rows: 449,359,707)
   Processed 110/282... (Rows: 494,069,295)
   Processed 120/282... (Rows: 538,420,933)
   Processed 130/282... (Rows: 584,435,608)
   Processed 140/282... (Rows: 629,613,787)
   Processed 150/282... (Rows: 673,615,446)
   Processed 160/282... (Rows: 718,269,988)
   Processed 170/282... (Rows: 763,346,515)
   Processed 180/282... (Rows: 808,342,793)
   Processed 190/282... (Rows: 854,558,912)
   Processed 200/282... (Rows: 900,235,819)
   Processed 210/282... (Rows: 944

In [3]:
import pandas as pd
import numpy as np
import os
import glob
import pyarrow.parquet as pq

# ==========================================
# CONFIGURATION
# ==========================================
# Auto-detect paths
BASE_DIR = "../data/processed" if os.path.exists("../data") else "data/processed"

DATASETS_TO_CHECK = {
    "PAR": {
        "path": os.path.join(BASE_DIR, "nasa_par_data_combined.parquet"),
        "col": "par",
        "valid_range": (0.0, 70.0),  # Einstein/m^2/day (Typical max is ~65)
        "unit": "Einstein/m²/day"
    },
    "Kd_490": {
        "path": os.path.join(BASE_DIR, "nasa_kd490_data_combined.parquet"),
        "col": "kd_490", # Note internal name
        "valid_range": (0.0, 6.0),   # 1/m (Turbid waters can go higher, but rare)
        "unit": "m^-1"
    },
    "POC": {
        "path": os.path.join(BASE_DIR, "nasa_poc_data_combined.parquet"),
        "col": "poc",
        "valid_range": (0.0, 1000.0), # mg/m^3 (Blooms can hit 500-1000)
        "unit": "mg/m³"
    }
}

# ==========================================
# AUDIT LOGIC
# ==========================================
def audit_dataset(name, config):
    print(f"\n==================================================")
    print(f"Checking Dataset: {name}")
    print(f"==================================================")
    
    path = config['path']
    col = config['col']
    
    if not os.path.exists(path):
        print(f"[ERROR] Path not found: {path}")
        return

    try:
        # 1. METADATA SCAN (Fast)
        # We use PyArrow Dataset to get total row count without loading
        dataset = pq.ParquetDataset(path)
        fragments = dataset.fragments
        num_partitions = len(fragments)
        
        print(f"Structure:        Partitioned Dataset")
        print(f"Partitions:       {num_partitions}")
        
        # 2. SAMPLING (Robust)
        # We take the first, middle, and last partition to check integrity
        indices = [0, num_partitions//2, num_partitions-1]
        sample_dfs = []
        
        print(f"Sampling {len(indices)} partitions for deep check...")
        
        files = sorted(glob.glob(os.path.join(path, "*.parquet")))
        
        for i in indices:
            if i < len(files):
                df_part = pd.read_parquet(files[i])
                sample_dfs.append(df_part)
        
        if not sample_dfs:
            print("[FAIL] No parquet files found inside folder.")
            return

        full_sample = pd.concat(sample_dfs)
        
        # 3. STATS CALCULATION
        print(f"Sample Size:      {len(full_sample):,} rows")
        
        # Date Range
        if 'time' in full_sample.columns:
            t_min = full_sample['time'].min()
            t_max = full_sample['time'].max()
            print(f"Time Range Sample:{t_min.date()} to {t_max.date()}")
        
        # Value Check
        if col not in full_sample.columns:
            # Fallback for naming mismatch
            print(f"[WARN] Column '{col}' not found. Available: {list(full_sample.columns)}")
            # Try to guess
            col = [c for c in full_sample.columns if c not in ['lat','lon','time']][0]
            print(f"       -> Using '{col}' instead.")

        val_min = full_sample[col].min()
        val_max = full_sample[col].max()
        val_mean = full_sample[col].mean()
        
        print(f"\nVariable '{col}' Stats ({config['unit']}):")
        print(f"   Min:  {val_min:.4f}")
        print(f"   Max:  {val_max:.4f}")
        print(f"   Mean: {val_mean:.4f}")
        
        # 4. LOGIC CHECK
        min_valid, max_valid = config['valid_range']
        
        # Check Min
        if val_min < -0.1: # Allow small float error around 0
            print(f">> [WARN] Found negative values! (Min: {val_min})")
        else:
            print(f">> [PASS] Min value is valid (>= 0).")
            
        # Check Max
        if val_max > max_valid:
            print(f">> [NOTE] Max value ({val_max}) exceeds typical range ({max_valid}).")
            print(f"          This is common for extreme outliers/blooms/glint.")
        else:
            print(f">> [PASS] Max value is within expected physical limits.")
            
        # Check NaNs
        nan_count = full_sample[col].isna().sum()
        if nan_count > 0:
            print(f">> [WARN] Found {nan_count} NaNs in sample.")
        else:
            print(f">> [PASS] Zero NaNs found in sample.")

    except Exception as e:
        print(f"[CRITICAL FAIL] Audit crashed: {e}")

def main():
    print("--- BlueEco: Multi-Variable Data Audit ---")
    
    for name, config in DATASETS_TO_CHECK.items():
        audit_dataset(name, config)
        
    print("\n--- Audit Complete ---")

if __name__ == "__main__":
    main()

--- BlueEco: Multi-Variable Data Audit ---

Checking Dataset: PAR
Structure:        Partitioned Dataset
Partitions:       282
Sampling 3 partitions for deep check...
Sample Size:      16,066,467 rows
Time Range Sample:2002-08-06 to 2026-01-01

Variable 'par' Stats (Einstein/m²/day):
   Min:  0.0020
   Max:  69.1980
   Mean: 34.4658
>> [PASS] Min value is valid (>= 0).
>> [PASS] Max value is within expected physical limits.
>> [PASS] Zero NaNs found in sample.

Checking Dataset: Kd_490
Structure:        Partitioned Dataset
Partitions:       282
Sampling 3 partitions for deep check...
Sample Size:      12,855,112 rows
Time Range Sample:2002-08-01 to 2026-01-01

Variable 'kd_490' Stats (m^-1):
   Min:  0.0166
   Max:  6.4000
   Mean: 0.0602
>> [PASS] Min value is valid (>= 0).
>> [NOTE] Max value (6.399999618530273) exceeds typical range (6.0).
          This is common for extreme outliers/blooms/glint.
>> [PASS] Zero NaNs found in sample.

Checking Dataset: POC
Structure:        Partitio